# Módulo 7 · Clase 9 — sklearn Pipelines y Despliegue de Modelos

**Curso de Inteligencia Artificial — Nivel básico**

---

## ¿Qué vamos a ver hoy?

| # | Tema |
|---|------|
| 1 | Por qué usar Pipelines |
| 2 | `Pipeline` básico |
| 3 | `ColumnTransformer` + Pipeline completo |
| 4 | Pipeline + GridSearchCV |
| 5 | Guardar y cargar modelos con `joblib` |

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
import os

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import (
    StandardScaler, MinMaxScaler, OneHotEncoder, OrdinalEncoder
)
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import (
    train_test_split, GridSearchCV, cross_val_score, StratifiedKFold
)
from sklearn.metrics import accuracy_score, classification_report

np.random.seed(42)
print("Librerías listas")

---
## 1. Por qué usar Pipelines

```
Sin Pipeline (código frágil y propenso a data leakage):

  scaler = StandardScaler()
  X_tr_sc = scaler.fit_transform(X_tr)   # OK: aprende parámetros en train
  X_te_sc = scaler.transform(X_te)       # OK: aplica los mismos
  modelo.fit(X_tr_sc, y_tr)
  modelo.predict(X_te_sc)
  # Problema: al hacer CV, tienes que repetir esto 5 veces manualmente
  # Error frecuente: hacer fit_transform(X_completo) antes de dividir → LEAKAGE

Con Pipeline:

  pipe = Pipeline([('scaler', StandardScaler()), ('clf', RandomForest())])
  pipe.fit(X_tr, y_tr)       # scaler.fit_transform(X_tr) + clf.fit
  pipe.predict(X_te)         # scaler.transform(X_te) + clf.predict
  cross_val_score(pipe, X, y, cv=5)  # garantiza NO leakage en cada fold
  GridSearchCV(pipe, param_grid)     # busca hiperparámetros SIN leakage

Ventajas del Pipeline:
  1. Garantiza orden correcto de pasos
  2. Previene data leakage en CV y GridSearch
  3. Guarda TODO el pipeline (scaler + modelo) en un solo fichero
  4. Una sola línea .predict() en producción
```

In [ ]:
from sklearn.datasets import make_classification

X, y = make_classification(n_samples=200, n_features=20, random_state=42)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)

# Forma INCORRECTA (leakage en CV)
scaler_mal = StandardScaler()
X_mal = scaler_mal.fit_transform(X)   # aprende media/std de TODO (incluye test)
scores_mal = cross_val_score(
    LogisticRegression(max_iter=500),
    X_mal, y, cv=5, scoring='accuracy'
)

# Forma CORRECTA (pipeline)
pipe_bien = Pipeline([
    ('scaler', StandardScaler()),
    ('clf',    LogisticRegression(max_iter=500))
])
scores_bien = cross_val_score(pipe_bien, X, y, cv=5, scoring='accuracy')

print("Comparativa leakage vs Pipeline:")
print(f"  Con leakage (score inflado): {scores_mal.mean():.4f} ± {scores_mal.std():.4f}")
print(f"  Sin leakage (Pipeline):      {scores_bien.mean():.4f} ± {scores_bien.std():.4f}")
print(f"  Diferencia: {scores_mal.mean() - scores_bien.mean():.4f} puntos de sesgo")

---
## 2. `Pipeline` básico

In [ ]:
# Anatomía de un Pipeline
pipe = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),   # paso 1: imputar NaN
    ('scaler',  StandardScaler()),                    # paso 2: escalar
    ('clf',     LogisticRegression(max_iter=1000)),   # paso final: modelo
])

print("Pasos del pipeline:")
for nombre, paso in pipe.steps:
    print(f"  [{nombre}] {type(paso).__name__}")

print(f"\nÚltimo estimador: {pipe[-1]}")

# Dataset con valores NaN para demostrar
X_nan = X.copy()
mascara = np.random.rand(*X_nan.shape) < 0.1  # 10% de NaN al azar
X_nan[mascara] = np.nan
X_tr_nan, X_te_nan, y_tr_n, y_te_n = train_test_split(X_nan, y, test_size=0.2, random_state=42)

pipe.fit(X_tr_nan, y_tr_n)
acc = accuracy_score(y_te_n, pipe.predict(X_te_nan))
print(f"\nAccuracy en test (datos con NaN): {acc:.4f}")

# Acceder a pasos internos
imputer_ajustado = pipe.named_steps['imputer']
print(f"\nMedianas aprendidas (5 primeras): {imputer_ajustado.statistics_[:5].round(3)}")

---
## 3. `ColumnTransformer` + Pipeline completo

```
ColumnTransformer permite aplicar transformaciones DISTINTAS a distintas columnas:

  Columnas numéricas → imputar (mediana) + escalar
  Columnas ordinales → imputar (más_frecuente) + OrdinalEncoder
  Columnas nominales → imputar (más_frecuente) + OneHotEncoder

  Todo en paralelo, sin data leakage, dentro del Pipeline.
```

In [ ]:
# Dataset realista con tipos mixtos
np.random.seed(42)
n = 500
df = pd.DataFrame({
    'edad':       np.random.randint(18, 70, n).astype(float),
    'salario':    np.random.exponential(30000, n) + 15000,
    'antiguedad': np.random.randint(0, 30, n).astype(float),
    'nivel':      np.random.choice(['junior','mid','senior'], n),
    'ciudad':     np.random.choice(['Madrid','Barcelona','Valencia','Bilbao'], n),
    'departamento': np.random.choice(['Tech','Sales','HR','Finance'], n),
})
# Introducir NaN
for col in ['edad','salario','nivel','ciudad']:
    idx = np.random.choice(n, 20, replace=False)
    df.loc[idx, col] = np.nan

# Target: promovido o no
y_df = ((df['antiguedad'].fillna(df['antiguedad'].median()) > 5) &
        (df['salario'].fillna(df['salario'].median()) > 30000)).astype(int)

print("Dataset (primeras filas):")
print(df.head())
print(f"\nNaN por columna:\n{df.isnull().sum()}")

In [ ]:
# Separar features por tipo
cols_numericas  = ['edad', 'salario', 'antiguedad']
cols_ordinales  = ['nivel']
cols_nominales  = ['ciudad', 'departamento']

X_df = df[cols_numericas + cols_ordinales + cols_nominales]
X_tr_df, X_te_df, y_tr_df, y_te_df = train_test_split(
    X_df, y_df, test_size=0.2, stratify=y_df, random_state=42
)

# Preprocesador por tipo de columna
prepro_numerico = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler()),
])

prepro_ordinal = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OrdinalEncoder(
        categories=[['junior','mid','senior']],
        handle_unknown='use_encoded_value', unknown_value=-1
    )),
])

prepro_nominal = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(sparse_output=False, drop='first', handle_unknown='ignore')),
])

# ColumnTransformer
preprocesador = ColumnTransformer(transformers=[
    ('num', prepro_numerico, cols_numericas),
    ('ord', prepro_ordinal,  cols_ordinales),
    ('nom', prepro_nominal,  cols_nominales),
])

# Pipeline final: preprocesador + modelo
pipeline_completo = Pipeline([
    ('prepro', preprocesador),
    ('clf',    RandomForestClassifier(100, random_state=42, n_jobs=-1)),
])

pipeline_completo.fit(X_tr_df, y_tr_df)
y_pred = pipeline_completo.predict(X_te_df)

print("Pipeline completo (ColumnTransformer + RandomForest):")
print(f"Accuracy en test: {accuracy_score(y_te_df, y_pred):.4f}")
print(f"\n{classification_report(y_te_df, y_pred, target_names=['No promovido','Promovido'])}") 

In [ ]:
# Inspeccionar la forma de los datos transformados
X_transformado = pipeline_completo.named_steps['prepro'].transform(X_te_df)
print(f"Shape original X_te: {X_te_df.shape}")
print(f"Shape transformado:   {X_transformado.shape}")

# Obtener nombres de features después de OHE
ohe = pipeline_completo.named_steps['prepro'].named_transformers_['nom'].named_steps['encoder']
nombres_ohe = ohe.get_feature_names_out(cols_nominales)

todos_nombres = cols_numericas + cols_ordinales + list(nombres_ohe)
print(f"\nFeatures después de transformar ({len(todos_nombres)}):")
print(todos_nombres)

# Feature importance
rf = pipeline_completo.named_steps['clf']
importancias = pd.Series(rf.feature_importances_, index=todos_nombres).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(9, 4))
importancias.head(10).plot(kind='barh', ax=ax, color='steelblue')
ax.set_xlabel('Importancia')
ax.set_title('Top 10 features más importantes')
plt.tight_layout(); plt.show()

---
## 4. Pipeline + GridSearchCV

In [ ]:
# Buscar hiperparámetros del preprocesador Y del modelo
# Sintaxis: 'nombrePaso__nombrePaso__hiperparametro' (doble __ para pasos anidados)

rejilla = {
    # Hiperparámetros del modelo
    'clf__n_estimators':        [50, 100, 200],
    'clf__max_depth':           [5, None],
    'clf__min_samples_split':   [2, 5],
    # Hiperparámetro del preprocesador (el scaler numérico)
    'prepro__num__scaler':      [StandardScaler(), MinMaxScaler()],
}

gs = GridSearchCV(
    pipeline_completo,
    rejilla,
    cv=StratifiedKFold(5, shuffle=True, random_state=42),
    scoring='accuracy',
    refit=True, n_jobs=-1, verbose=1
)
gs.fit(X_tr_df, y_tr_df)

print(f"\nMejores parámetros: {gs.best_params_}")
print(f"Score CV: {gs.best_score_:.4f}")
print(f"Accuracy test: {accuracy_score(y_te_df, gs.predict(X_te_df)):.4f}")

---
## 5. Guardar y cargar modelos con `joblib`

```
Por qué guardar el modelo:
  - Reentrenar es caro (tiempo, GPU, datos)
  - El servidor de producción no ejecuta el entrenamiento
  - Versionar modelos: saber qué modelo generó qué predicción

joblib vs pickle:
  joblib es más eficiente con arrays numpy grandes (internos de sklearn)
  pickle para objetos Python pequeños
  En sklearn: usa siempre joblib

IMPORTANTE: guarda el PIPELINE completo (preprocesador + modelo)
  Al cargarlo, solo necesitas .predict(X_raw) → el pipeline hace todo el preprocesamiento.
```

In [ ]:
# 1. Guardar el pipeline completo
ruta_modelo = 'modelo_promovidos_v1.joblib'
joblib.dump(gs.best_estimator_, ruta_modelo)
print(f"Modelo guardado en: {ruta_modelo}")
print(f"Tamaño del fichero: {os.path.getsize(ruta_modelo) / 1024:.1f} KB")

# 2. Cargar el modelo
modelo_cargado = joblib.load(ruta_modelo)
print(f"\nModelo cargado: {type(modelo_cargado).__name__}")

# 3. Usar el modelo cargado (datos RAW sin preprocesar)
acc_cargado = accuracy_score(y_te_df, modelo_cargado.predict(X_te_df))
print(f"Accuracy del modelo cargado: {acc_cargado:.4f}")

# 4. Simular datos de producción (un cliente nuevo, con posibles NaN)
cliente_nuevo = pd.DataFrame({
    'edad':           [32],
    'salario':        [45000],
    'antiguedad':     [7],
    'nivel':          ['mid'],
    'ciudad':         ['Madrid'],
    'departamento':   ['Tech'],
})

prediccion = modelo_cargado.predict(cliente_nuevo)[0]
proba      = modelo_cargado.predict_proba(cliente_nuevo)[0]
print(f"\nCliente nuevo:")
print(cliente_nuevo.to_string(index=False))
print(f"\nPredicción: {'Promovido' if prediccion==1 else 'No promovido'}")
print(f"Probabilidad: {proba[1]:.4f} de ser promovido")

In [ ]:
# Buenas prácticas: guardar metadatos junto al modelo
import json

metadatos = {
    'version':          'v1.0.0',
    'features_entrada': cols_numericas + cols_ordinales + cols_nominales,
    'target':           'promovido (0=No, 1=Si)',
    'algoritmo':        'RandomForestClassifier',
    'mejores_params':   {k: str(v) for k, v in gs.best_params_.items()},
    'score_cv':         round(gs.best_score_, 4),
    'accuracy_test':    round(acc_cargado, 4),
    'n_train':          len(X_tr_df),
}

ruta_meta = 'modelo_promovidos_v1_meta.json'
with open(ruta_meta, 'w', encoding='utf-8') as f:
    json.dump(metadatos, f, indent=2, ensure_ascii=False)

print("Metadatos guardados:")
print(json.dumps(metadatos, indent=2, ensure_ascii=False))

---
## Práctica

### Ejercicio 1 — Pipeline con Titanic

**Enunciado:** Usa el dataset Titanic de seaborn:

```python
import seaborn as sns
titanic = sns.load_dataset('titanic')
features = ['pclass', 'sex', 'age', 'sibsp', 'parch', 'fare', 'embarked']
target = 'survived'
```

1. Construye un Pipeline completo con `ColumnTransformer`:
   - Numéricas: `age`, `sibsp`, `parch`, `fare` → imputar (mediana) + StandardScaler
   - Ordinales: `pclass` → OrdinalEncoder
   - Nominales: `sex`, `embarked` → imputar (más_frecuente) + OneHotEncoder
2. Añade un RandomForestClassifier como último paso
3. Evalúa con 5-Fold CV
4. Muestra accuracy, F1 y las 5 features más importantes

In [ ]:
# ✏️ TU CÓDIGO AQUÍ


<details>
<summary>💡 Ver solución — Ejercicio 1</summary>

```python
import seaborn as sns
import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OrdinalEncoder, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_validate, StratifiedKFold

titanic = sns.load_dataset('titanic').dropna(subset=['survived'])
features = ['pclass','sex','age','sibsp','parch','fare','embarked']
X = titanic[features]; y = titanic['survived']

cols_num  = ['age','sibsp','parch','fare']
cols_ord  = ['pclass']
cols_nom  = ['sex','embarked']

prepro_num = Pipeline([('imp', SimpleImputer(strategy='median')), ('sc', StandardScaler())])
prepro_ord = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
prepro_nom = Pipeline([
    ('imp', SimpleImputer(strategy='most_frequent')),
    ('ohe', OneHotEncoder(sparse_output=False, drop='first', handle_unknown='ignore')),
])

ct = ColumnTransformer([
    ('num', prepro_num, cols_num),
    ('ord', prepro_ord, cols_ord),
    ('nom', prepro_nom, cols_nom),
])

pipe = Pipeline([
    ('prepro', ct),
    ('clf',    RandomForestClassifier(100, class_weight='balanced', random_state=42, n_jobs=-1)),
])

cv = StratifiedKFold(5, shuffle=True, random_state=42)
res = cross_validate(pipe, X, y, cv=cv, scoring={'acc':'accuracy','f1':'f1'}, return_train_score=True)
print(f"Accuracy CV: {res['test_acc'].mean():.4f} ± {res['test_acc'].std():.4f}")
print(f"F1 CV:       {res['test_f1'].mean():.4f} ± {res['test_f1'].std():.4f}")

# Feature importances
pipe.fit(X, y)
rf = pipe.named_steps['clf']
ohe_names = pipe.named_steps['prepro'].named_transformers_['nom'].named_steps['ohe'].get_feature_names_out(cols_nom)
nombres = cols_num + cols_ord + list(ohe_names)
imp = pd.Series(rf.feature_importances_, index=nombres).sort_values(ascending=False)
print(f"\nTop 5 features: {imp.head(5).index.tolist()}")
```
</details>

---
### Ejercicio 2 — Comparar 3 modelos dentro del mismo Pipeline

**Enunciado:** Usa el dataset de diabetes:

```python
from sklearn.datasets import load_diabetes
X, y = load_diabetes(return_X_y=True)
# Binarizar target (encima o debajo de la mediana)
y = (y > np.median(y)).astype(int)
```

1. Define un Pipeline base: `StandardScaler → clf`
2. Prueba 3 clasificadores distintos con cross_val_score:
   - LogisticRegression
   - RandomForestClassifier(100)
   - GradientBoostingClassifier(100)
3. Muestra una tabla comparativa con accuracy y F1
4. ¿Cuál elegirías y por qué?

In [ ]:
# ✏️ TU CÓDIGO AQUÍ


<details>
<summary>💡 Ver solución — Ejercicio 2</summary>

```python
import numpy as np
import pandas as pd
from sklearn.datasets import load_diabetes
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import cross_validate, StratifiedKFold

X, y = load_diabetes(return_X_y=True)
y = (y > np.median(y)).astype(int)

cv = StratifiedKFold(5, shuffle=True, random_state=42)
filas = []
for nombre, clf in [
    ('LogisticRegression', LogisticRegression(max_iter=5000, random_state=42)),
    ('RandomForest',       RandomForestClassifier(100, random_state=42, n_jobs=-1)),
    ('GradientBoosting',   GradientBoostingClassifier(100, random_state=42)),
]:
    pipe = Pipeline([('sc', StandardScaler()), ('clf', clf)])
    res = cross_validate(pipe, X, y, cv=cv, scoring={'acc':'accuracy','f1':'f1'}, n_jobs=-1)
    filas.append({
        'Modelo':   nombre,
        'Acc_media': res['test_acc'].mean(),
        'Acc_std':   res['test_acc'].std(),
        'F1_media':  res['test_f1'].mean(),
        'F1_std':    res['test_f1'].std(),
    })

df_comp = pd.DataFrame(filas).sort_values('F1_media', ascending=False)
print(df_comp.round(4).to_string(index=False))
print("\nElección sugerida: GradientBoosting suele ganar en datasets tabulares medianos.")
print("Para datasets grandes, RandomForest puede ser más rápido.")
```
</details>

---
### Ejercicio 3 — GridSearch completo

**Enunciado:** Usa el Pipeline del Ejercicio 1 (Titanic) y optimiza sus hiperparámetros:

1. Define una rejilla que explore:
   - `clf__n_estimators`: [50, 100, 200]
   - `clf__max_depth`: [5, 10, None]
   - `clf__class_weight`: ['balanced', None]
2. Usa `GridSearchCV` con 5 folds
3. Compara el modelo por defecto vs el optimizado
4. ¿Cuánto mejoró el F1?

In [ ]:
# ✏️ TU CÓDIGO AQUÍ


<details>
<summary>💡 Ver solución — Ejercicio 3</summary>

```python
import seaborn as sns
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_val_score
# Reutilizar pipe y datos del ejercicio 1

titanic = sns.load_dataset('titanic').dropna(subset=['survived'])
features = ['pclass','sex','age','sibsp','parch','fare','embarked']
X, y = titanic[features], titanic['survived']

# (reutilizar pipe del ejercicio 1...)
# Asegúrate de tener pipe definido antes de ejecutar esto

rejilla = {
    'clf__n_estimators':  [50, 100, 200],
    'clf__max_depth':     [5, 10, None],
    'clf__class_weight':  ['balanced', None],
}

cv = StratifiedKFold(5, shuffle=True, random_state=42)

# Baseline
f1_base = cross_val_score(pipe, X, y, cv=cv, scoring='f1').mean()

# GridSearch
gs = GridSearchCV(pipe, rejilla, cv=cv, scoring='f1', refit=True, n_jobs=-1)
gs.fit(X, y)

print(f"F1 baseline (por defecto): {f1_base:.4f}")
print(f"F1 optimizado (CV):        {gs.best_score_:.4f}")
print(f"Mejora: +{gs.best_score_ - f1_base:.4f}")
print(f"Mejores parámetros: {gs.best_params_}")
```
</details>

---
### Ejercicio 4 — Ciclo completo: guardar y cargar

**Enunciado:** Simula el ciclo completo de producción:

1. Entrena el pipeline del Ejercicio 1 en el 80% de los datos
2. Guárdalo con `joblib.dump` como `'titanic_modelo.joblib'`
3. Guarda sus metadatos en `'titanic_modelo_meta.json'`
4. Cárgalo de nuevo con `joblib.load`
5. Genera predicciones para 3 pasajeros ficticios:
   - Pasajero A: pclass=1, sex='female', age=25, sibsp=0, parch=0, fare=200, embarked='C'
   - Pasajero B: pclass=3, sex='male', age=35, sibsp=0, parch=0, fare=7, embarked='S'
   - Pasajero C: pclass=2, sex='male', age=10, sibsp=1, parch=2, fare=30, embarked='Q'
6. Muestra la predicción y la probabilidad de supervivencia

In [ ]:
# ✏️ TU CÓDIGO AQUÍ


<details>
<summary>💡 Ver solución — Ejercicio 4</summary>

```python
import joblib, json
import pandas as pd
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

titanic = sns.load_dataset('titanic').dropna(subset=['survived'])
features = ['pclass','sex','age','sibsp','parch','fare','embarked']
X, y = titanic[features], titanic['survived']
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

# Usar pipe del ejercicio 1 ya definido
pipe.fit(X_tr, y_tr)
acc_te = accuracy_score(y_te, pipe.predict(X_te))

# Guardar
joblib.dump(pipe, 'titanic_modelo.joblib')
meta = {
    'features': features,
    'target': 'survived (0=No, 1=Si)',
    'accuracy_test': round(acc_te, 4),
    'n_train': len(X_tr),
}
with open('titanic_modelo_meta.json', 'w') as f:
    json.dump(meta, f, indent=2)
print("Modelo guardado.")

# Cargar y predecir
modelo = joblib.load('titanic_modelo.joblib')
pasajeros = pd.DataFrame([
    {'pclass':1,'sex':'female','age':25,'sibsp':0,'parch':0,'fare':200,'embarked':'C'},
    {'pclass':3,'sex':'male',  'age':35,'sibsp':0,'parch':0,'fare':7,  'embarked':'S'},
    {'pclass':2,'sex':'male',  'age':10,'sibsp':1,'parch':2,'fare':30, 'embarked':'Q'},
], columns=features)

for i, (_, row) in enumerate(pasajeros.iterrows()):
    X_row = pd.DataFrame([row])
    pred  = modelo.predict(X_row)[0]
    proba = modelo.predict_proba(X_row)[0][1]
    print(f"Pasajero {chr(65+i)}: {'Superviviente' if pred==1 else 'No superviviente'} "
          f"(prob={proba:.3f})")
```
</details>

---
### Ejercicio 5 (reto) — Fábrica de Pipelines

**Enunciado:** Implementa `crear_pipeline(modelo, cols_num, cols_ord, cols_nom, ord_cats=None)` que:

1. Crea automáticamente el ColumnTransformer para los 3 tipos de columnas
2. Añade el modelo al final
3. Devuelve el Pipeline listo para usar

Pruébala creando 3 pipelines distintos (RandomForest, LogisticRegression, GradientBoosting) con el mismo ColumnTransformer y compáralos con `cross_val_score`.

In [ ]:
# ✏️ TU CÓDIGO AQUÍ


<details>
<summary>💡 Ver solución — Ejercicio 5</summary>

```python
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OrdinalEncoder, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

def crear_pipeline(modelo, cols_num=None, cols_ord=None, cols_nom=None, ord_cats=None):
    transformers = []
    if cols_num:
        transformers.append(('num', Pipeline([
            ('imp', SimpleImputer(strategy='median')),
            ('sc',  StandardScaler()),
        ]), cols_num))
    if cols_ord:
        cats = ord_cats if ord_cats else 'auto'
        transformers.append(('ord', Pipeline([
            ('imp', SimpleImputer(strategy='most_frequent')),
            ('enc', OrdinalEncoder(categories=cats,
                                   handle_unknown='use_encoded_value', unknown_value=-1)),
        ]), cols_ord))
    if cols_nom:
        transformers.append(('nom', Pipeline([
            ('imp', SimpleImputer(strategy='most_frequent')),
            ('ohe', OneHotEncoder(sparse_output=False, drop='first', handle_unknown='ignore')),
        ]), cols_nom))

    ct = ColumnTransformer(transformers)
    return Pipeline([('prepro', ct), ('clf', modelo)])

# Probar con Titanic
import seaborn as sns
titanic = sns.load_dataset('titanic').dropna(subset=['survived'])
features = ['pclass','sex','age','sibsp','parch','fare','embarked']
X, y = titanic[features], titanic['survived']

cv = StratifiedKFold(5, shuffle=True, random_state=42)
modelos = [
    ('RandomForest',     RandomForestClassifier(100, class_weight='balanced', random_state=42, n_jobs=-1)),
    ('LogisticReg',      LogisticRegression(max_iter=5000, random_state=42)),
    ('GradientBoosting', GradientBoostingClassifier(100, random_state=42)),
]

print(f"{'Modelo':22} {'Acc CV':>10} {'F1 CV':>10}")
print("-" * 44)
for nombre, clf in modelos:
    pipe = crear_pipeline(
        clf,
        cols_num=['age','sibsp','parch','fare'],
        cols_ord=['pclass'],
        cols_nom=['sex','embarked'],
    )
    acc = cross_val_score(pipe, X, y, cv=cv, scoring='accuracy', n_jobs=-1).mean()
    f1  = cross_val_score(pipe, X, y, cv=cv, scoring='f1',       n_jobs=-1).mean()
    print(f"{nombre:22} {acc:>10.4f} {f1:>10.4f}")
```
</details>

---
## Resumen

```python
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
import joblib

# Pipeline básico
pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler()),
    ('clf',     RandomForestClassifier(100)),
])

# ColumnTransformer para tipos mixtos
ct = ColumnTransformer([
    ('num', Pipeline([('imp',SimpleImputer()), ('sc',StandardScaler())]), cols_num),
    ('ord', OrdinalEncoder(), cols_ord),
    ('nom', OneHotEncoder(sparse_output=False, drop='first'), cols_nom),
])
pipe_full = Pipeline([('prepro', ct), ('clf', RandomForestClassifier())])

# GridSearchCV sobre pipeline
gs = GridSearchCV(pipe_full, {'clf__n_estimators': [50,100]}, cv=5)
gs.fit(X_tr, y_tr)      # acceso: 'nombrePaso__hiperparametro'

# Guardar / cargar
joblib.dump(gs.best_estimator_, 'modelo.joblib')   # guarda TODO el pipeline
modelo = joblib.load('modelo.joblib')
pred = modelo.predict(X_nuevo_raw)                  # sin preprocesar manualmente
```

## ¿Qué viene en la próxima clase?
En la **Clase 10** pondremos todo en práctica con un **Proyecto final end-to-end** usando el dataset de California Housing: EDA, pipeline completo, múltiples modelos, GridSearchCV e informe final.